# 业务背景

某消费品牌经过全年经营，积累了订单、用户、商品、渠道、会员、促销及售后等多维业务数据。为全面了解企业全年经营状况，识别销售增长与下滑的关键驱动因素，发现渠道、商品及用户经营中的潜在问题，基于全年订单数据开展销售经营分析。

通过对销售趋势、订单规模、客单价、渠道、商品、用户、会员、促销及售后等维度进行分析，定位业务增长点与风险点，并结合分析结果提出针对性的运营优化建议。

## 指标口径速查

阅读本文前请先确认口径。完整版见文末《指标口径说明》。

| 指标 | 口径 |
|---|---|
| 销售额 | 商品单价 × 购买数量，**毛口径未扣退货**（全年 136.25 万元，扣除退货后净额约 126.76 万元） |
| 客单价 | 销售额 ÷ 订单量 |
| 连带率 | 销售件数 ÷ 订单量 |
| 退货率 | 退货订单**笔数** ÷ 订单笔数 = 731/10,354 = 7.06%（金额口径 6.97%） |
| 换货 | 仅统计笔数，**未做金额调整** |
| 复购用户 | 购买次数 > 1 的用户（**用户级**标签） |
| 首单/非首单 | 数据集"首次下单用户"字段（**订单级**标记） |
| RFM-M | 用户累计消费金额，毛口径 |

## 核心问题

2023年销售表现如何？有什么潜在问题？

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pyecharts.charts import Bar, Line, Pie, Grid
from pyecharts import options as opts

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))   # 自动向上查找 .env
except ImportError:
    pass   # 没装 python-dotenv 也能跑，直接读系统环境变量

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST', '127.0.0.1').split(':')[0]
DB_PORT     = os.getenv('DB_PORT', '3306')
DB_NAME     = os.getenv('DB_NAME', 'order_analysis')

missing = [k for k, v in {
    'DB_USER': DB_USER, 'DB_PASSWORD': DB_PASSWORD,
    'DB_HOST': DB_HOST, 'DB_NAME': DB_NAME}.items() if not v]
if missing:
    raise RuntimeError(
        f'缺少环境变量：{missing}\n'
        f'请复制 .env.example 为 .env 并填写，再执行 python init_db.py 导入数据。'
    )

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}'
    f'/{DB_NAME}?charset=utf8mb4'
)

In [2]:
sql = """
SELECT
    COUNT(*) AS 订单量,
    COUNT(DISTINCT 用户ID) AS 用户数,
    SUM(购买数量) AS 销售件数,
    SUM(商品单价 * 购买数量) AS 销售额
FROM orders_detail
"""

overview = pd.read_sql(sql, engine)

overview['客单价'] = (
    overview['销售额'] / overview['订单量']
).round(1)

overview

,订单量,用户数,销售件数,销售额,客单价
0,10354,5137,12137.0,1362543.0,131.6


### 月度销售趋势

In [3]:
sql1 = """
SELECT
    DATE_FORMAT(订单交易时间, '%%Y-%%m') AS 年月,
    COUNT(DISTINCT 用户ID) AS 用户数,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额,
    SUM(购买数量) AS 件数
FROM orders_detail
GROUP BY 1
ORDER BY 1
"""

m = pd.read_sql(sql1, engine)

m['客单价'] = (
    m['销售额'] / m['订单量']
).round(1)

m['连带率'] = (
    m['件数'] / m['订单量']
).round(2)

m['销售额环比'] = (
    m['销售额'].pct_change() * 100
).round(1)

m['订单量环比'] = (
    m['订单量'].pct_change() * 100
).round(1)

m['客单价环比'] = (
    m['客单价'].pct_change() * 100
).round(1)

m

,年月,用户数,订单量,销售额,件数,客单价,连带率,销售额环比,订单量环比,客单价环比
0,2023-01,393,480,63199.0,561.0,131.7,1.17,NaN,NaN,NaN
1,2023-02,289,324,43791.0,389.0,135.2,1.20,-30.7,-32.5,2.7
2,2023-03,541,663,90319.0,781.0,136.2,1.18,106.3,104.6,0.7
3,2023-04,513,607,80824.0,716.0,133.2,1.18,-10.5,-8.4,-2.2
4,2023-05,573,679,92253.0,807.0,135.9,1.19,14.1,11.9,2.0
5,2023-06,971,1272,170867.0,1513.0,134.3,1.19,85.2,87.3,-1.2
6,2023-07,568,678,87319.0,781.0,128.8,1.15,-48.9,-46.7,-4.1
7,2023-08,682,822,106157.0,963.0,129.1,1.17,21.6,21.2,0.2
8,2023-09,569,692,86955.0,785.0,125.7,1.13,-18.1,-15.8,-2.6
9,2023-10,755,911,114932.0,1048.0,126.2,1.15,32.2,31.6,0.4


In [4]:
(Line()
    .add_xaxis(m['年月'].tolist())
    .add_yaxis(
        '销售额',
        m['销售额'].round(2).tolist()
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title='月度销售额趋势')
    )
 .render_notebook()
)

In [5]:
 (Bar()
    .add_xaxis(m['年月'].tolist())
    .add_yaxis(
        '订单量',
        m['订单量'].tolist()
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title='月度订单量趋势')
    )
    .render_notebook()
 )

### 计算客单价、订单量、销售额环比

In [6]:
sql2 = """
SELECT
    DATE_FORMAT(订单交易时间, '%%Y-%%m') AS 年月,
    COUNT(DISTINCT 用户ID) AS 用户数,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额
FROM orders_detail
GROUP BY 1
ORDER BY 1
"""

sales_decompose = pd.read_sql(sql2, engine)

In [7]:
sales_decompose['客单价'] = (
    sales_decompose['销售额'] /
    sales_decompose['订单量']
).round(1)

sales_decompose['订单量环比'] = (
    sales_decompose['订单量'].pct_change() * 100
).round(1)

sales_decompose['客单价环比'] = (
    sales_decompose['客单价'].pct_change() * 100
).round(1)

sales_decompose['销售额环比'] = (
    sales_decompose['销售额'].pct_change() * 100
).round(1)

sales_decompose

,年月,用户数,订单量,销售额,客单价,订单量环比,客单价环比,销售额环比
0,2023-01,393,480,63199.0,131.7,NaN,NaN,NaN
1,2023-02,289,324,43791.0,135.2,-32.5,2.7,-30.7
2,2023-03,541,663,90319.0,136.2,104.6,0.7,106.3
3,2023-04,513,607,80824.0,133.2,-8.4,-2.2,-10.5
4,2023-05,573,679,92253.0,135.9,11.9,2.0,14.1
5,2023-06,971,1272,170867.0,134.3,87.3,-1.2,85.2
6,2023-07,568,678,87319.0,128.8,-46.7,-4.1,-48.9
7,2023-08,682,822,106157.0,129.1,21.2,0.2,21.6
8,2023-09,569,692,86955.0,125.7,-15.8,-2.6,-18.1
9,2023-10,755,911,114932.0,126.2,31.6,0.4,32.2


In [8]:
dates = (sales_decompose['年月'].str[-2:].astype(int).astype(str) + '月').tolist()
line = (
    Line(init_opts=opts.InitOpts(width="1200px", height="500px"))
    .add_xaxis(dates)
    .add_yaxis("订单量环比", sales_decompose['订单量环比'].tolist())
    .add_yaxis("客单价环比", sales_decompose['客单价环比'].tolist())
    .add_yaxis("销售额环比", sales_decompose['销售额环比'].tolist())
    .set_global_opts(
        title_opts=opts.TitleOpts(title="三个指标环比趋势"),
        yaxis_opts=opts.AxisOpts(name="环比（%）"),
        legend_opts=opts.LegendOpts(pos_top="5%")
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(
            is_show=True, formatter="{c}%"
)
    )
)

line.render_notebook()

其中11月销售额达到全年高点26.96万元，相比10月环比增长134.57%。
是由于双十一活动大促提升了销售额和订单量，不作为异常波动
销售额增长幅度与订单量增长幅度高度接近，而客单价变化较小。

### 渠道分析

In [9]:
sql3 = """
SELECT
    订单来源,
    COUNT(DISTINCT 用户ID) AS 用户数,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额,
    SUM(购买数量) AS 件数
FROM orders_detail
GROUP BY 订单来源
ORDER BY 销售额 DESC
"""

channel = pd.read_sql(sql3, engine)
channel['客单价'] = (
    channel['销售额'] /
    channel['订单量']
).round(1)

channel['连带率'] = (
    channel['件数'] /
    channel['订单量']
).round(2)

channel['销售额占比'] = (
    channel['销售额'] /
    channel['销售额'].sum() * 100
).round(2)

channel

,订单来源,用户数,订单量,销售额,件数,客单价,连带率,销售额占比
0,店铺下单,3918,6891,928077.0,8203.0,134.7,1.19,68.11
1,直播下单,2317,3463,434466.0,3934.0,125.5,1.14,31.89


In [10]:
pie_data = [list(z) for z in zip(channel['订单来源'], channel['销售额占比'])]

# 左：柱形图（沿用你的写法）
bar = (
    Bar()
    .add_xaxis(channel['订单来源'].tolist())
    .add_yaxis(
        '销售额',
        channel['销售额'].round(2).tolist()
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title='渠道销售额对比'),
        yaxis_opts=opts.AxisOpts(name="销售额（元）"),
        legend_opts=opts.LegendOpts(pos_left="15%")
        
    )
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True, formatter="{c}"))
)

# 右：环形图
pie = (
    Pie()
    .add("", pie_data, radius=["40%", "70%"],center=["65%", "50%"])
    .set_global_opts(
        title_opts=opts.TitleOpts(title='各渠道销售额占比', pos_right="30%"),
        legend_opts=opts.LegendOpts(orient="vertical", pos_right="15%")
    )
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}: {c}%"))
)

# 用Grid左右并排
grid = (
    Grid(init_opts=opts.InitOpts(width="1000px", height="500px"))
    .add(bar, grid_opts=opts.GridOpts(pos_left="5%", pos_right="65%", pos_top="15%", pos_bottom="10%"))
    .add(pie, grid_opts=opts.GridOpts())
)

# 显示
grid.render_notebook()

### 渠道×月份

In [11]:
sql4 = """
SELECT
    DATE_FORMAT(订单交易时间, '%%Y-%%m') AS 年月,
    订单来源,
    SUM(商品单价 * 购买数量) AS 销售额
FROM orders_detail
GROUP BY 1, 2
ORDER BY 1, 2
"""

channel_month = pd.read_sql(sql4, engine)

channel_month_pivot = channel_month.pivot_table(
    index='年月',
    columns='订单来源',
    values='销售额',
    fill_value = 0
)

channel_month_pivot

订单来源,店铺下单,直播下单
年月,,
2023-01,43760,19439
2023-02,32020,11771
2023-03,64467,25852
2023-04,58660,22164
2023-05,61060,31193
2023-06,115207,55660
2023-07,59047,28272
2023-08,74314,31843
2023-09,58945,28010


### 渠道分析结论

全年店铺渠道销售额约92.81万元，占全年销售额68.11%。

直播渠道销售额约43.45万元，占比31.88%。

店铺渠道承担主要销售规模，同时客单价高于直播渠道。


### 品类分析

In [12]:
sql5 = """
SELECT
    品类,
    COUNT(DISTINCT 用户ID) AS 用户数,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额,
    SUM(购买数量) AS 件数,
    SUM(退货) AS 退货订单
FROM orders_detail
GROUP BY 品类
ORDER BY 销售额 DESC
"""

category = pd.read_sql(sql5, engine)
category['客单价'] = (
    category['销售额'] /
    category['订单量']
).round(1)

category['连带率'] = (
    category['件数'] /
    category['订单量']
).round(2)

category['销售额占比'] = (
    category['销售额'] /
    category['销售额'].sum() * 100
).round(2)

category['退货率'] = (
    category['退货订单'] /
    category['订单量'] * 100
).round(2)

category

,品类,用户数,订单量,销售额,件数,退货订单,客单价,连带率,销售额占比,退货率
0,A,2752,4285,592977.0,4983.0,268.0,138.4,1.16,43.52,6.25
1,B,1294,1762,426457.0,2143.0,119.0,242.0,1.22,31.30,6.75
2,C,1422,1994,187467.0,2373.0,173.0,94.0,1.19,13.76,8.68
3,D,1654,2313,155642.0,2638.0,171.0,67.3,1.14,11.42,7.39


In [13]:
bar = (
    Bar()
    .add_xaxis(category['品类'].tolist())
    .add_yaxis('订单量', category['订单量'].round(2).tolist())
    .set_global_opts(
        title_opts=opts.TitleOpts(title='各品类订单量'),
        yaxis_opts=opts.AxisOpts(name="订单量"),
        legend_opts=opts.LegendOpts(pos_left="20%")

    )
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True, formatter="{c}"))
)

# 环形图
pie_data = [list(z) for z in zip(category['品类'], category['销售额占比'])]
pie = (
    Pie()
    .add("", pie_data, radius=["40%", "70%"],center=["65%", "50%"])
    .set_global_opts(
        title_opts=opts.TitleOpts(title='各品类销售额占比', pos_right="30%"),
        legend_opts=opts.LegendOpts(orient="vertical", pos_right="10%")

    )
    .set_series_opts(label_opts=opts.LabelOpts(formatter="{b}: {c}%"))
)

# 左右并排
grid = Grid(init_opts=opts.InitOpts(width="1000px", height="500px"))
grid.add(bar, grid_opts=opts.GridOpts(pos_left="5%", pos_right="60%"))
grid.add(pie, grid_opts=opts.GridOpts())
grid.render_notebook()

### 品类分析结论

A品类 是基本盘（占 43.5% 销售额、退货率最低 6.25%）
B品类是高价值商品（客单价 242，17% 订单贡献 31.3% 销售额，值得加投放）
C品类 退货率最高（8.68%）需要优先排查
D品类 走量不走价（22% 订单只换 11.4% 销售额，投产比最差）

### 是否复购用户分析

In [14]:
sql6 = """
WITH 用户标签 AS (
    SELECT DISTINCT 用户ID,
           CASE WHEN COUNT(*) OVER (PARTITION BY 用户ID) > 1
                THEN '复购用户' ELSE '仅首单用户' END AS 用户类型
    FROM orders_detail
)
SELECT
    t.用户类型,
    COUNT(DISTINCT o.用户ID)     AS 用户数,
    COUNT(*)                     AS 订单量,
    SUM(o.购买数量)              AS 件数,
    SUM(o.商品单价 * o.购买数量) AS 销售额
FROM orders_detail o
JOIN 用户标签 t ON o.用户ID = t.用户ID
GROUP BY t.用户类型
"""

new_old = pd.read_sql(sql6, engine)

new_old['客单价'] = (new_old['销售额'] / new_old['订单量']).round(1)
new_old['连带率'] = (new_old['件数'] / new_old['订单量']).round(2)
new_old['销售额占比'] = (new_old['销售额'] / new_old['销售额'].sum() * 100).round(2)

new_old

,用户类型,用户数,订单量,件数,销售额,客单价,连带率,销售额占比
0,仅首单用户,2588,2588,3038.0,343602.0,132.8,1.17,25.22
1,复购用户,2549,7766,9099.0,1018941.0,131.2,1.17,74.78


### 复购分析结论

复购是生意的基本盘——一半用户（2,549 人）产生复购，贡献了 74.8% 的销售额；另一半 2,588 人买一次就走，仅占 25.2%，复购用户人均件数达到了3.57

### 新老客×渠道

In [15]:
sql7 = """
SELECT
    订单来源,
    首次下单用户,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额
FROM orders_detail
GROUP BY 订单来源, 首次下单用户
"""

new_old_channel = pd.read_sql(sql7, engine)

new_old_channel_pivot = pd.pivot_table(
    new_old_channel,
    index='订单来源',
    columns='首次下单用户',
    values='销售额',
    fill_value=0
)

new_old_channel_pivot

首次下单用户,否,是
订单来源,,
店铺下单,460609,467468
直播下单,224917,209549


In [16]:
from pyecharts.charts import Bar
from pyecharts import options as opts

# 准备数据
channels = new_old_channel_pivot.index.tolist()
# '首次下单用户' 字段：'是' = 新客首单，'否' = 老客复购
new_sales = new_old_channel_pivot['是'].tolist()  
old_sales = new_old_channel_pivot['否'].tolist()  

bar = (
    Bar()
    .add_xaxis(channels)
    .add_yaxis("新客", new_sales, stack="stack1",bar_width='40%')
    .add_yaxis("老客", old_sales, stack="stack1",bar_width='40%')
    .set_global_opts(
        title_opts=opts.TitleOpts(title="各渠道新老客销售额分布"),
        xaxis_opts=opts.AxisOpts(name="渠道"),
        yaxis_opts=opts.AxisOpts(name="销售额（元）"),
        legend_opts=opts.LegendOpts(pos_top="5%")
    )
    .set_series_opts(label_opts=opts.LabelOpts(is_show=True, position="inside"))
)

bar.render_notebook()

店铺是主渠道：占 68.11% 销售额，直播占 31.9%
新客和老客的销售额占比接近1:1

### 会员分析

In [17]:
sql8 = """
with 最新状态 as(
            select 用户ID,会员,
                    row_number() over(
                    partition by 用户ID
                    order by 订单交易时间 desc,订单编号 desc
                    ) rn
            from orders_detail
            ) 
SELECT
    s.会员,
    COUNT(DISTINCT o.用户ID)     AS 用户数,
    COUNT(*)                     AS 订单量,
    SUM(o.购买数量)              AS 件数,
    SUM(o.商品单价 * o.购买数量) AS 销售额,
    SUM(o.退货)                  AS 退货订单
FROM orders_detail o
JOIN 最新状态 s
  ON o.用户ID = s.用户ID AND s.rn = 1
GROUP BY s.会员
"""

member = pd.read_sql(sql8, engine) 

member['客单价'] = (member['销售额'] / member['订单量']).round(2)
member['连带率'] = (member['件数'] / member['订单量']).round(2)
member['退货率'] = (member['退货订单'] / member['订单量'] * 100).round(2)
member['销售额占比'] = (member['销售额'] / member['销售额'].sum() * 100).round(2)

member

,会员,用户数,订单量,件数,销售额,退货订单,客单价,连带率,退货率,销售额占比
0,否,3435,6146,7166.0,805314.0,426.0,131.03,1.17,6.93,59.1
1,是,1702,4208,4971.0,557229.0,305.0,132.42,1.18,7.25,40.9


### 会员分析结论

非会员销售额占 59.1%，订单量 6146，会员仅占 40.9%。
会员客单价（132.4）略高于非会员（131.0），且人均下单频次更高（会员2.47单/人 vs 非会员1.79单/人）
会员退货率（7.25%）比非会员高（6.93%）

### 活动分析

In [18]:
sql9 = """
SELECT
    活动优惠,
    COUNT(DISTINCT 用户ID) AS 用户数,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额,
    SUM(购买数量) AS 件数,
    SUM(退货) AS 退货订单
FROM orders_detail
GROUP BY 活动优惠
"""

promotion = pd.read_sql(sql9, engine)

promotion['客单价'] = (promotion['销售额'] /promotion['订单量']).round(1)
promotion['退货率'] = (promotion['退货订单'] /promotion['订单量'] * 100).round(2)
promotion['连带率'] = promotion['件数']/promotion['订单量']
promotion['销售额占比'] = (promotion['销售额'] /promotion['销售额'].sum() * 100).round(2)

promotion

,活动优惠,用户数,订单量,销售额,件数,退货订单,客单价,退货率,连带率,销售额占比
0,无,4788,9258,1233025.0,10935.0,677.0,133.2,7.31,1.181141,90.49
1,有,807,1096,129518.0,1202.0,54.0,118.2,4.93,1.096715,9.51


In [19]:
from pyecharts import options as opts
from pyecharts.charts import Bar, Page

p = promotion.set_index('活动优惠')

metrics = [
    ("客单价(元)", float(p.loc['无', '客单价']), float(p.loc['有', '客单价'])),
    ("退货率(%)",  float(p.loc['无', '退货率']), float(p.loc['有', '退货率'])),
    ("连带率",    float(p.loc['无', '连带率']), float(p.loc['有', '连带率'])),
]

page = Page()
for name, v_no, v_yes in metrics:
    b = (
        Bar(init_opts=opts.InitOpts(width="880px", height="260px"))
        .add_xaxis([name])
        .add_yaxis("无优惠", [v_no], label_opts=opts.LabelOpts(formatter="{c}", position="top"))
        .add_yaxis("有优惠", [v_yes], label_opts=opts.LabelOpts(formatter="{c}", position="top"))
        .set_global_opts(title_opts=opts.TitleOpts(title=f"无优惠 vs 有优惠 · {name}", pos_left="left"))
    )
    page.add(b)

page.render_notebook()

### 活动分析结论

优惠单体量小：订单占 10.6%、销售额仅占 9.51%，但用户占 15.7%
优惠单客单价：118.2 vs 133.2 元,
优惠单退货率更低：4.93% vs 7.31%

### 活动 × 新老客

In [20]:
sql10 = """
SELECT
    活动优惠,
    首次下单用户,
    COUNT(*) AS 订单量,
    SUM(商品单价 * 购买数量) AS 销售额
FROM orders_detail
GROUP BY 活动优惠, 首次下单用户
"""

promotion_new_old = pd.read_sql(sql10, engine)

promotion_new_old_pivot = pd.pivot_table(
    promotion_new_old,
    index='活动优惠',
    columns='首次下单用户',
    values='销售额',
    fill_value=0
)

promotion_new_old_pivot

首次下单用户,否,是
活动优惠,,
无,616994,616031
有,68532,60986


### 活动 × 新老客分析结论

首次下单用户和非首次下单用户的销售额接近1:1

### 售后分析

In [21]:
sql11 = """
SELECT
    品类,
    COUNT(*) AS 订单量,
    SUM(退货) AS 退货订单,
    SUM(换货) AS 换货订单
FROM orders_detail
GROUP BY 品类
"""

after_sale = pd.read_sql(sql11, engine)

after_sale['退货率'] = (
    after_sale['退货订单'] /
    after_sale['订单量'] * 100
).round(2)

after_sale['换货率'] = (
    after_sale['换货订单'] /
    after_sale['订单量'] * 100
).round(2)

after_sale.sort_values(
    '退货率',
    ascending=False
)

,品类,订单量,退货订单,换货订单,退货率,换货率
2,C,1994,173.0,24.0,8.68,1.20
1,D,2313,171.0,36.0,7.39,1.56
0,B,1762,119.0,34.0,6.75,1.93
3,A,4285,268.0,69.0,6.25,1.61


In [22]:
from pyecharts import options as opts
from pyecharts.charts import Pie

# 数据准备
data = [
    (row['品类'], row['退货率'])
    for _, row in after_sale.sort_values('退货率', ascending=False).iterrows()
]

# 画图
c = (
    Pie()
    .add("", data, radius=["40%", "70%"])  # 核心：环形
    .set_global_opts(
        title_opts=opts.TitleOpts(title="各品类退货率环形图")
    )
    .set_series_opts(
        label_opts=opts.LabelOpts(formatter="{b}: {c}%")  # 标签在这里设置
    )
)

c.render_notebook()

### 售后分析结论

A退货率最低，但绝对损失最大：退货率 6.25% 为四品类最低，可 43.5% 的销售额体量让它贡献了 37,604 元退货金额，占全场退货总额的 39.6%，是 C 的 2.1 倍。

B高客单叠加高换货，尺码/规格类问题代价最重：换货率 1.93% 为四品类最高，叠加 242 元的最高客单价，单次尺码或规格不合的往返物流与处置成本都是最重的，且换货不产生销售额。加投放前需先确认尺码/规格信息准确，否则销量放大等于把这块成本等比放大。

C退货重灾区：退货率 8.68% 最高，退货金额吃掉自身销售额 9.61%，是纯粹的经营性损耗品类。

D走量不走价：22.3% 的订单（第二多）只换回 11.4% 销售额（最少），客单价 67.3 元与连带率 1.14 双低，且不带动其他品类。

注：四品类连带率仅 1.14–1.22，差异不构成有效结论；11月大促是普涨、不改变品类结构。


### RFM用户分层

In [23]:
sql12 = """
SELECT
    用户ID,
    MAX(订单交易时间) AS 最近购买时间,
    COUNT(*) AS 购买次数,
    SUM(商品单价 * 购买数量) AS 消费金额
FROM orders_detail
GROUP BY 用户ID
"""

user_rfm = pd.read_sql(sql12, engine)

user_rfm['最近购买时间'] = pd.to_datetime(
    user_rfm['最近购买时间']
)

snapshot_date = (
    user_rfm['最近购买时间'].max()
    + pd.Timedelta(days=1)
)

user_rfm['R'] = (
    snapshot_date -
    user_rfm['最近购买时间']
).dt.days
user_rfm

,用户ID,最近购买时间,购买次数,消费金额,R
0,YS26659,2023-01-01 09:41:57,1,199.0,365
1,HN401006,2023-12-18 08:20:20,9,1249.0,14
2,YS08625,2023-02-21 21:54:15,2,258.0,314
3,YS82117,2023-01-16 17:33:58,2,138.0,350
4,YS08454,2023-01-01 17:55:50,1,59.0,365
...,...,...,...,...,...
5132,TM948977,2023-12-31 10:54:56,1,59.0,1
5133,YS14876,2023-12-31 13:33:16,1,119.0,1
5134,IB139749,2023-12-31 14:24:45,1,79.0,1
5135,YS08352,2023-12-31 16:56:16,1,119.0,1


RFM评分

In [24]:
# R：天数连续、无大量并列，保留四分位
user_rfm['R得分'] = pd.qcut(user_rfm['R'], 4, labels=[4, 3, 2, 1]).astype(int) 

# F：50.4%用户仅1单，四分位会随机拆并列值 → 改用业务锚点分箱
# 1单=未复购，2单=完成首复购，3-4单=习惯形成，5单+=高频
user_rfm['F得分'] = pd.cut(user_rfm['购买次数'],
                           bins=[0, 1, 2, 4, float('inf')],
                           labels=[1, 2, 3, 4]).astype(int)

# M：金额连续，保留四分位
user_rfm['M得分'] = pd.qcut(user_rfm['消费金额'], 4, labels=[1, 2, 3, 4]).astype(int)

user_rfm[['用户ID', 'R', '购买次数', '消费金额', 'R得分', 'F得分', 'M得分']].head(10)


,用户ID,R,购买次数,消费金额,R得分,F得分,M得分
0,YS26659,365,1,199.0,1,1,3
1,HN401006,14,9,1249.0,4,4,4
2,YS08625,314,2,258.0,1,2,3
3,YS82117,350,2,138.0,1,2,2
4,YS08454,365,1,59.0,1,1,1
5,YS49395,363,2,278.0,1,2,3
6,YS24398,329,2,198.0,1,2,2
7,YS31139,365,1,119.0,1,1,1
8,YS70576,50,2,138.0,4,2,2
9,FM719415,52,2,398.0,3,2,4


用户分层

In [25]:
# 各维度与自身均值比较定高低 → 经典八分层（阈值随数据自适应）
r_hi = user_rfm['R得分'] >= user_rfm['R得分'].mean()
f_hi = user_rfm['F得分'] >= user_rfm['F得分'].mean()
m_hi = user_rfm['M得分'] >= user_rfm['M得分'].mean()

user_rfm['用户分层'] = np.select(
    [r_hi & f_hi & m_hi, ~r_hi & f_hi & m_hi, ~r_hi & ~f_hi & m_hi,
     r_hi & ~f_hi & m_hi, r_hi & f_hi & ~m_hi, r_hi & ~f_hi & ~m_hi,
     ~r_hi & f_hi & ~m_hi],
    ['重要价值客户', '重要保持客户', '重要挽留客户','重要发展客户',
              '一般价值客户', '一般发展客户', '一般保持客户'],
    default='一般挽留客户')


user_rfm[['用户ID', 'R得分', 'F得分', 'M得分', '用户分层']].head(10)

,用户ID,R得分,F得分,M得分,用户分层
0,YS26659,1,1,3,重要挽留客户
1,HN401006,4,4,4,重要价值客户
2,YS08625,1,2,3,重要保持客户
3,YS82117,1,2,2,一般保持客户
4,YS08454,1,1,1,一般挽留客户
5,YS49395,1,2,3,重要保持客户
6,YS24398,1,2,2,一般保持客户
7,YS31139,1,1,1,一般挽留客户
8,YS70576,4,2,2,一般价值客户
9,FM719415,3,2,4,重要价值客户


In [26]:
tier_order = ['重要价值客户', '重要保持客户', '重要挽留客户','重要发展客户',
              '一般价值客户', '一般发展客户', '一般保持客户', '一般挽留客户']
user_rfm['用户分层'] = pd.Categorical(user_rfm['用户分层'], categories=tier_order, ordered=True)

user_level_full = (
    user_rfm.groupby("用户分层",as_index=False).agg(
    用户数 = ("用户ID" , "count"),
    平均购买次数 = ("购买次数", "mean"),
    平均消费金额 = ("消费金额", "mean")
    )
)

user_level_full["用户占比"] = (user_level_full["用户数"]/user_level_full["用户数"].sum()*100).round(2)

user_level_full


,用户分层,用户数,平均购买次数,平均消费金额,用户占比
0,重要价值客户,1290,3.671318,510.589922,25.11
1,重要保持客户,632,2.751582,409.591772,12.30
2,重要挽留客户,403,1.000000,253.310174,7.85
3,重要发展客户,222,1.000000,245.689189,4.32
4,一般价值客户,328,2.073171,162.298780,6.39
5,一般发展客户,738,1.000000,95.771003,14.37
6,一般保持客户,299,2.043478,161.150502,5.82
7,一般挽留客户,1225,1.000000,94.935510,23.85


命名规律：
前缀看 M：M 高 = "重要"，M 低 = "一般"；
后缀看 R×F：R 高 F 高 =价值、R 低 F 高 =保持、R 低 F 低 =挽留、R 高 F 低 =发展。

In [27]:
from pyecharts.charts import Pie

pie = (
    Pie()
    .add(
        "",
        data_pair=list(zip(user_level_full["用户分层"], user_level_full["用户数"])),
        radius=["40%", "70%"],  # 环形图
        label_opts=opts.LabelOpts(
            formatter="{b}: {d}%",
            position="outside"
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="用户层级占比"),
        legend_opts=opts.LegendOpts(
            orient="vertical",
            pos_top="center",
            pos_right="5%"
        )
    )
    .set_series_opts(
        tooltip_opts=opts.TooltipOpts(
            formatter="{b}<br/>用户数: {c}<br/>占比: {d}%"
        )
    )
)

pie.render_notebook()

### 留出验证：用上半年数据分层，检验对下半年消费的预测力

In [28]:

h1 ="""
    SELECT
    用户ID,
    MAX(订单交易时间) AS 最近购买时间,
    COUNT(*) AS 购买次数,
    SUM(商品单价 * 购买数量) AS 消费金额
FROM orders_detail
WHERE 订单交易时间 < '2023-07-01'
GROUP BY 用户ID
"""
# …再用同样的分层逻辑打标签，merge 下半年消费做 groupby 对比

user_rfm_h1 = pd.read_sql(h1,engine)

user_rfm_h1['最近购买时间'] = pd.to_datetime(
    user_rfm_h1['最近购买时间']
)

snapshot_date_h1 = (
    user_rfm_h1['最近购买时间'].max()
    + pd.Timedelta(days=1)
)

user_rfm_h1['R'] = (
    snapshot_date_h1 -
    user_rfm_h1['最近购买时间']
).dt.days

In [29]:
# ---- H1 打分：qcut 分位，同时保存切点供 H2 复用 ----
# R：天数连续无并列，保留四分位
user_rfm_h1['R得分'] = pd.qcut(user_rfm_h1['R'], 4, labels=[4, 3, 2, 1]).astype(int)
# # F：50.4%用户仅1单，四分位会随机拆并列值 → 改用业务锚点分箱
# # 锚点依据：1单=未复购，2单=完成首复购，3-4单=习惯形成，5单+=高频
user_rfm_h1['F得分'] = pd.cut(user_rfm_h1['购买次数'],
                             bins=[0, 1, 2, 4, float('inf')],
                             labels=[1, 2, 3, 4]).astype(int)
user_rfm_h1['M得分'] = pd.qcut(user_rfm_h1['消费金额'], 4, labels=[1, 2, 3, 4]).astype(int)

r_bins = pd.qcut(user_rfm_h1['R'], 4, retbins=True)[1].copy()
m_bins = pd.qcut(user_rfm_h1['消费金额'], 4, retbins=True)[1].copy()
# 两端放开，否则 H2 有超出范围的样本会变 NaN
r_bins[0], r_bins[-1] = -np.inf, np.inf
m_bins[0], m_bins[-1] = -np.inf, np.inf

# ---- 分层阈值统一取 H1 的均值，保证两期可比 ----
R_THR = user_rfm_h1['R得分'].mean()
F_THR = user_rfm_h1['F得分'].mean()
M_THR = user_rfm_h1['M得分'].mean()


def assign_tier(df):
    r_hi = df['R得分'] >= R_THR
    f_hi = df['F得分'] >= F_THR
    m_hi = df['M得分'] >= M_THR
    return np.select(
        [r_hi & f_hi & m_hi, ~r_hi & f_hi & m_hi, ~r_hi & ~f_hi & m_hi,
         r_hi & ~f_hi & m_hi, r_hi & f_hi & ~m_hi, r_hi & ~f_hi & ~m_hi,
         ~r_hi & f_hi & ~m_hi],
        ['重要价值客户', '重要保持客户', '重要挽留客户', '重要发展客户',
         '一般价值客户', '一般发展客户', '一般保持客户'],
        default='一般挽留客户')


user_rfm_h1['用户分层'] = assign_tier(user_rfm_h1)
user_rfm_h1.head()

,用户ID,最近购买时间,购买次数,消费金额,R,R得分,F得分,M得分,用户分层
0,YS26659,2023-01-01 09:41:57,1,199.0,181,1,1,3,重要挽留客户
1,HN401006,2023-06-16 12:54:37,7,931.0,15,4,4,4,重要价值客户
2,YS08625,2023-02-21 21:54:15,2,258.0,130,1,2,4,重要保持客户
3,YS82117,2023-01-16 17:33:58,2,138.0,166,1,2,2,一般保持客户
4,YS08454,2023-01-01 17:55:50,1,59.0,181,1,1,1,一般挽留客户


In [30]:
tier_order = ['重要价值客户', '重要保持客户', '重要挽留客户','重要发展客户',
              '一般价值客户', '一般发展客户', '一般保持客户', '一般挽留客户']
user_rfm_h1['用户分层'] = pd.Categorical(user_rfm_h1['用户分层'], categories=tier_order, ordered=True)

user_level_h1 = (
    user_rfm_h1.groupby("用户分层",as_index=False).agg(
    用户数 = ("用户ID" , "count"),
    平均购买次数 = ("购买次数", "mean"),
    平均消费金额 = ("消费金额", "mean")
    )
)

user_level_h1["用户占比"] = (user_level_h1["用户数"]/user_level_h1["用户数"].sum()*100).round(2)

user_level_h1

,用户分层,用户数,平均购买次数,平均消费金额,用户占比
0,重要价值客户,508,2.921260,415.547244,20.36
1,重要保持客户,289,2.446367,345.439446,11.58
2,重要挽留客户,222,1.000000,256.504505,8.90
3,重要发展客户,176,1.000000,247.295455,7.05
4,一般价值客户,68,2.000000,136.235294,2.73
5,一般发展客户,503,1.000000,95.141153,20.16
6,一般保持客户,68,2.000000,134.470588,2.73
7,一般挽留客户,661,1.000000,96.204236,26.49


下半年用户分层

In [31]:
# ---- H2 数据读取：与 H1 的 < '2023-07-01' 严丝合缝，>= 不漏 7/1 当天订单 ----
h2 = """
    SELECT
    用户ID,
    MAX(订单交易时间) AS 最近购买时间,
    COUNT(*) AS 购买次数,
    SUM(商品单价 * 购买数量) AS 消费金额
FROM orders_detail
WHERE 订单交易时间 >= '2023-07-01'
GROUP BY 用户ID
"""

user_rfm_h2 = pd.read_sql(h2, engine)

user_rfm_h2['最近购买时间'] = pd.to_datetime(
    user_rfm_h2['最近购买时间']
)

# H2 自己的快照日 = 下半年最大交易时间 + 1（与 H1 各算各的期末新鲜度）
snapshot_date_h2 = (
    user_rfm_h2['最近购买时间'].max()
    + pd.Timedelta(days=1)
)

user_rfm_h2['R'] = (
    snapshot_date_h2 -
    user_rfm_h2['最近购买时间']
).dt.days

user_rfm_h2.head()

,用户ID,最近购买时间,购买次数,消费金额,R
0,YS80336,2023-08-08 19:03:10,2,138.0,146
1,BM188861,2023-10-13 11:54:19,3,397.0,80
2,NH490380,2023-08-07 18:09:52,2,158.0,147
3,GO566,2023-11-11 23:26:30,2,357.0,51
4,YS19214,2023-11-03 21:05:18,2,178.0,59


In [32]:
# ---- H2 用 H1 的切点打分，保证"重要价值客户"在两期含义一致 ----
user_rfm_h2['R得分'] = pd.cut(user_rfm_h2['R'], bins=r_bins,
                             labels=[4, 3, 2, 1], include_lowest=True).astype(int)
user_rfm_h2['F得分'] = pd.cut(user_rfm_h2['购买次数'],
                             bins=[0, 1, 2, 4, float('inf')],
                             labels=[1, 2, 3, 4]).astype(int)
user_rfm_h2['M得分'] = pd.cut(user_rfm_h2['消费金额'], bins=m_bins,
                             labels=[1, 2, 3, 4], include_lowest=True).astype(int)

user_rfm_h2['用户分层'] = assign_tier(user_rfm_h2)

user_rfm_h2.head(10)

,用户ID,最近购买时间,购买次数,消费金额,R,R得分,F得分,M得分,用户分层
0,YS80336,2023-08-08 19:03:10,2,138.0,146,1,2,2,一般保持客户
1,BM188861,2023-10-13 11:54:19,3,397.0,80,2,3,4,重要保持客户
2,NH490380,2023-08-07 18:09:52,2,158.0,147,1,2,2,一般保持客户
3,GO566,2023-11-11 23:26:30,2,357.0,51,2,2,4,重要保持客户
4,YS19214,2023-11-03 21:05:18,2,178.0,59,2,2,3,重要保持客户
5,YS72177,2023-07-18 20:02:14,2,278.0,167,1,2,4,重要保持客户
6,HL1023,2023-10-30 13:39:04,2,278.0,63,2,2,4,重要保持客户
7,YS51059,2023-07-01 20:10:41,1,119.0,184,1,1,2,一般挽留客户
8,YS94620,2023-07-01 20:24:04,1,59.0,184,1,1,1,一般挽留客户
9,YS23442,2023-07-01 21:13:39,1,199.0,184,1,1,3,重要挽留客户


In [33]:
tier_order = ['重要价值客户', '重要保持客户', '重要挽留客户','重要发展客户',
              '一般价值客户', '一般发展客户', '一般保持客户', '一般挽留客户']
user_rfm_h2['用户分层'] = pd.Categorical(user_rfm_h2['用户分层'], categories=tier_order, ordered=True)

user_level_h2 = (
    user_rfm_h2.groupby("用户分层",as_index=False).agg(
    用户数 = ("用户ID" , "count"),
    平均购买次数 = ("购买次数", "mean"),
    平均消费金额 = ("消费金额", "mean")
    )
)

user_level_h2["用户占比"] = (user_level_h2["用户数"]/user_level_h2["用户数"].sum()*100).round(2)

user_level_h2

,用户分层,用户数,平均购买次数,平均消费金额,用户占比
0,重要价值客户,768,3.522135,469.246094,23.64
1,重要保持客户,595,2.608403,354.721008,18.31
2,重要挽留客户,314,1.000000,241.659236,9.66
3,重要发展客户,104,1.000000,265.432692,3.20
4,一般价值客户,85,2.000000,135.411765,2.62
5,一般发展客户,345,1.000000,97.733333,10.62
6,一般保持客户,101,2.000000,136.811881,3.11
7,一般挽留客户,937,1.000000,93.188901,28.84


对比

In [34]:
# 上半年：只保留用户ID + 分层 + R/F/M得分
h1_keep = user_rfm_h1[['用户ID', '用户分层', 'R得分', 'F得分', 'M得分']].rename(
    columns={
        '用户分层': 'H1分层',
        'R得分': 'H1_R得分',
        'F得分': 'H1_F得分',
        'M得分': 'H1_M得分'
    }
)

## 下半年：保留分层 + 消费结果
h2_keep = user_rfm_h2[['用户ID', '用户分层', '消费金额', '购买次数']].rename(
    columns={
        '用户分层': 'H2分层',
        '消费金额': 'H2消费金额',
        '购买次数': 'H2购买次数'
    }
)

user_rfm_h1_h2 = h1_keep.merge(h2_keep, on='用户ID', how='left')
user_rfm_h1_h2['H2消费金额'] = user_rfm_h1_h2['H2消费金额'].fillna(0)
user_rfm_h1_h2['H2购买次数'] = user_rfm_h1_h2['H2购买次数'].fillna(0)
# 上半年有、下半年一次没买的人 → 流失
user_rfm_h1_h2['H2分层'] = (
    user_rfm_h1_h2['H2分层'].astype(str).replace('nan', '未复购')
)
user_rfm_h1_h2

,用户ID,H1分层,H1_R得分,H1_F得分,H1_M得分,H2分层,H2消费金额,H2购买次数
0,YS26659,重要挽留客户,1,1,3,未复购,0.0,0.0
1,HN401006,重要价值客户,4,4,4,重要价值客户,318.0,2.0
2,YS08625,重要保持客户,1,2,4,未复购,0.0,0.0
3,YS82117,一般保持客户,1,2,2,未复购,0.0,0.0
4,YS08454,一般挽留客户,1,1,1,未复购,0.0,0.0
...,...,...,...,...,...,...,...,...
2490,YS64754,一般发展客户,4,1,1,未复购,0.0,0.0
2491,VZ221720,一般发展客户,4,1,1,未复购,0.0,0.0
2492,YS24317,一般发展客户,4,1,2,重要挽留客户,199.0,1.0
2493,YS54236,一般发展客户,4,1,1,重要挽留客户,199.0,1.0


In [35]:
# 迁移矩阵：行=上半年分层，列=下半年分层；数值=行内占比(%)
migration_pct = (
    pd.crosstab(user_rfm_h1_h2['H1分层'],
                user_rfm_h1_h2['H2分层'],
                normalize='index') * 100
)

# 列按八分层固定顺序，“未复购”统一置末
migration_pct = (
    migration_pct
    .reindex(columns=tier_order + ['未复购'])
    .fillna(0)
    .round(1)
)

migration_pct


H2分层,重要价值客户,重要保持客户,重要挽留客户,重要发展客户,一般价值客户,一般发展客户,一般保持客户,一般挽留客户,未复购
H1分层,,,,,,,,,
重要价值客户,8.9,10.8,4.7,1.4,0.4,3.1,1.8,13.4,55.5
重要保持客户,2.4,1.4,3.5,0.0,0.3,2.4,0.3,9.0,80.6
重要挽留客户,0.9,0.9,3.2,1.8,0.5,1.8,0.0,1.4,89.6
重要发展客户,2.8,9.1,5.1,2.8,0.6,1.1,1.7,12.5,64.2
一般价值客户,2.9,4.4,4.4,0.0,0.0,2.9,1.5,16.2,67.6
一般发展客户,3.4,8.2,3.6,0.2,0.4,2.2,1.0,11.7,69.4
一般保持客户,0.0,1.5,2.9,0.0,0.0,1.5,0.0,4.4,89.7
一般挽留客户,0.2,0.2,0.9,0.3,0.2,1.5,0.0,5.3,91.5


In [36]:
# 按上半年分层，看下半年平均消费和购买次数
result = user_rfm_h1_h2.groupby('H1分层').agg(
    用户数=('用户ID', 'count'),
    下半年平均消费=('H2消费金额', 'mean'),
    下半年平均购买次数=('H2购买次数', 'mean'),
    下半年总消费=('H2消费金额', 'sum')
).reset_index()

# 按下半年平均消费降序，看分层是否单调
result = result.sort_values('下半年平均消费', ascending=False)
print(result)

     H1分层  用户数     下半年平均消费  下半年平均购买次数   下半年总消费
0  重要价值客户  508  125.246063   0.913386  63625.0
3  重要发展客户  176   90.517045   0.619318  15931.0
5  一般发展客户  503   70.137177   0.572565  35279.0
4  一般价值客户   68   53.529412   0.544118   3640.0
1  重要保持客户  289   30.906574   0.262976   8932.0
2  重要挽留客户  222   22.531532   0.126126   5002.0
6  一般保持客户   68   18.661765   0.132353   1269.0
7  一般挽留客户  661   11.095310   0.089259   7334.0


### 留出验证结论

- **分层对下期消费有预测力**：H1 重要价值客户下半年人均消费 125.2 元、0.91 次，是一般挽留客户（11.1 元、0.09 次）的约 11 倍，排序大体单调。
- **唯一倒挂的是重要保持客户**：下半年人均仅 30.9 元、80.6% 未复购。该层在 H1 时 R 已偏低（高频高金额但近期未购），说明 **R（近期活跃度）比 F/M 更能预测下期消费**，这类 "沉默老客" 召回难度大，不宜按 F/M 高就过度投入。
- **整体留存偏弱**：H1 活跃的 2,495 人中，各层下半年未复购比例为 55.5%~91.5%。
- **验证范围**：仅覆盖 H1 活跃的 2,495 人；H2 共 3,249 人，两期均活跃 607 人，H2 新客不在本验证内。

### RFM 八层的差异化运营动作

资源排序**不按 H1 的表面价值，而按 H2 的实际产出**——这正是留出验证的意义。

| 分层 | 人数（占比） | H2 实际表现 | 运营动作 | 优先级 |
|---|---|---|---|---|
| **重要价值客户** | 508（20.4%） | 人均 125.2 元**最高**，但仍有 55.5% 未复购 | 专属权益、新品优先体验、VIP 通道。目标是压降这 55.5% 的流失——最好的一层过半没回来，是全盘最大漏洞 | 高 |
| **重要发展客户** | 176（7.1%） | 人均 90.5 元**第 2**，未复购 64.2% | **投产比最高**。最近来过、金额高但只买 1 次 → 首单后 30 天内二次触达 + 门槛型优惠，目标是提频 | 高 |
| **一般发展客户** | 503（20.2%） | 人均 70.1 元**第 3**，未复购 69.4% | 盘子第二大且最近活跃。自动化触达 + 组合优惠，观察向上迁移率，**不投人工成本** | 中 |
| **一般价值客户** | 68（2.7%） | 人均 53.5 元 | 买得多但单价低 → 交叉销售高客单品类（B 品类 242 元）提客单价 | 中 |
| **重要保持客户** | 289（11.6%） | 人均仅 **30.9 元**，**80.6% 未复购** | ⚠️ **陷阱层**：H1 高频高金额极易被误判为核心，实际已大面积沉默。先用低成本券测试唤醒意愿，**有响应再升级投入**，不要凭 F/M 高就上重资源 | 谨慎 |
| **重要挽留客户** | 222（8.9%） | 人均 22.5 元，89.6% 未复购 | 曾经高消费、现已沉寂。一次性召回 + 流失原因调研，不投长期权益 | 低 |
| **一般保持客户** | 68（2.7%） | 人均 18.7 元 | 人数仅占 2.7%，不做专项，纳入常规自动化池 | 低 |
| **一般挽留客户** | 661（**26.5%**） | 人均 **11.1 元垫底**，91.5% 未复购 | 人数最多、产出最低的三低人群。**&#8203;只保留低成本自动触达，不投人工与付费资源** | 暂不投入 |

**为什么这样排**：如果只看 H1 表面价值，"重要保持客户"排在前列；但 H2 验证显示它人均只有 30.9 元、80.6% 没回来——**是留出验证把它从第 2 位拽到了第 5 位**。同理，"一般发展客户"表面是低价值层，实际 H2 人均排第 3，因此给了中优先级。

没有留出验证，这两层的资源都会配反。


## 指标口径说明

| 指标 | 定义 | 备注 |
|---|---|---|
| 销售额（GMV） | 商品单价 × 购买数量 | 毛口径，未扣退货。全年 136.25 万元 |
| 净销售额 | 毛销售额 − 退货订单金额 | 约 126.76 万元（退货约 9.50 万元） |
| 订单量 | orders_detail 行数 | 10,354 单 |
| 用户数 | COUNT(DISTINCT 用户ID) | 5,137 人 |
| 客单价 | 销售额 ÷ 订单量 | |
| 连带率 | 销售件数 ÷ 订单量 | |
| 退货率 | 退货订单笔数 ÷ 订单笔数 | 订单数口径，731/10,354 = 7.06% |
| 退货率（金额口径） | 退货订单金额 ÷ 总金额 | 6.97%，与笔数口径有差异 |
| 换货 | 仅统计笔数 | 未做金额调整 |
| 复购用户 | 购买次数 > 1 的用户 | 用户级标签，2,549 人贡献 74.8% 销售额 |
| 首单 / 非首单 | 数据集中"首次下单用户"字段 | 订单级标记，收入结构约 1:1 |
| R | 快照日 − 最近购买时间（天） | 快照日 = 数据内最大交易日期 + 1 |
| F | 购买次数 | 业务锚点分箱：1/2/3-4/5+ |
| M | 累计消费金额 | 毛口径 |
|重要|M 高 = "重要"||
|一般|M 低 = "一般"||

### 已知局限
- 新老客存在用户级与订单级两套口径，数值不可混用：用户级看复购人群贡献（74.8%），订单级看首单收入结构（约 1:1）。
- 会员状态取用户最新一单的取值，中途入会用户的历史订单一并归入会员组。
- 换货订单未做金额调整。
- 数据为和鲸社区公开脱敏数据集，分析结论用于方法论验证，不作为真实业务判断。


## 结论与行动建议

### 一、全局判断

全年生意由**复购人群**和**店铺渠道**两根柱子撑着：2,549 名复购用户贡献 74.8% 销售额，店铺渠道贡献 68.11%。增长主要靠订单量驱动而非客单价，且高度依赖 11 月大促（单月 26.96 万元、环比 +134.57%），淡旺季波动剧烈。

因此运营重心应放在**守住现有复购盘 + 提高非大促月份的产出**，而不是继续加大促力度。

### 二、行动建议（按优先级）

**1. 守住复购盘 —— 最高优先级**

- 依据：2,549 名复购用户（约占用户数一半）贡献 74.8% 销售额，人均件数 3.57；另外 2,588 人只买一次即流失
- 动作：把资源从"拉新"切一部分到"首单后 30 天内的二次触达"，对已产生 2 次购买的用户做会员/权益绑定，防止其在第 3 单前流失
- 验证：按首单月份分组，对比触达组与未触达组的 90 日内二次购买率；观察 RFM 中"重要保持客户"（高频高金额但近期未购）的回流比例

**2. B 品类加大投放（需先解决履约成本）**

- 依据：客单价 242 元，用 17.0% 的订单换回 31.3% 的销售额，投产比显著优于其他品类
- 动作：在店铺渠道首页与直播间主推位给 B 品类更多曝光；用 A 的稳定复购人群做关联推荐带动 B 的首次尝试（不给 A 打折引流）
- 前置条件：① B 换货率 1.93% 为四品类最高，需先核算单件换货履约成本是否在可接受区间；
  ② B 仅覆盖 1,294 人，加投放前需确认目标人群规模足以承接
- 验证：投放前后对比 B 的订单占比与换货率，若换货率同步上升则暂停扩量，先做尺码/规格信息优化


**3. C 品类退货优先排查，定性为"描述不符"而非"尺码问题"**

- 依据：退货率 8.68% 为四品类最高，退货金额吃掉 9.61% 销售额；
  但换货率仅 1.20% 为四品类最低——退货高、换货低，
  说明用户收货后直接放弃而非要求调换，指向详情页描述与实物落差或质量预期问题
- 动作：拉取 C 品类退货订单明细按 SKU 归类，同时对比详情页图文与实物差异；
  补充实物图、真实场景图与规格参数表，而非仅补充尺码对照表
- 验证：改动上线后跟踪 C 品类退货率月度变化，目标向 A 品类（6.25%）收敛


**4. 会员权益从"折扣"转向"频次激励"**&#8203;

- 依据：会员人均 2.47 单 / 非会员 1.79 单，频次优势明显；但客单价几乎无差异（132.4 vs 131.0），且会员退货率反而更高（7.25% vs 6.93%）
- 动作：现有会员权益对客单价无效，建议改成"累计次数阶梯权益"（如第 3 单解锁免邮 / 第 5 单解锁专享价），直接激励频次
- 验证：对比改版前后会员的人均下单次数与退货率；若退货率同步下降，说明折扣型权益带来的冲动购买被抑制

**5. 优惠单以"控退货"为目标，暂不扩直降**

- 依据：优惠单客单价低 11.3%（118.2 vs 133.2），连带率（1.10 vs 1.18）、人均件数（1.49 vs 2.28）也都略低——优惠并没有换来多件；新客销售额占比 47.1% 与无优惠（50.0%）接近，也无明显拉新效果。唯一明确优势是退货率显著更低（4.93% vs 7.31%）
- 动作：不做无门槛直降扩量；若目标是提连带率，改测"满 2 件折 / 满额包邮"等门槛型优惠；优惠保留在新客首单等小范围场景，主打低退货体验
- 验证：A/B 对比门槛型优惠与直降的连带率、退货率和 90 日复购率，指标明确改善再扩量


**6. 平滑淡旺季，降低大促依赖**

- 依据：11 月单月环比 +134.57%，意味着其余月份产出被大促虹吸
- 动作：在 3–4 月、7–8 月等低谷期做小规模品类专场，测试非大促时段的承接能力
- 验证：对比活动月的销售额环比与非活动月的自然波动，评估低谷期是否真的能抬升基线

### 三、暂不建议投入

| 对象 | 原因 |
|---|---|
| D 品类 | 22.3% 的订单（第二多）只换来 11.4% 销售额（最少），客单价 67.3 元最低、连带率 1.14 最低，退货率 7.39% 还高于 A/B。走量不走价且不带动其他品类，应压缩曝光资源 |
| 一般挽留 / 一般保持客户 | 留出验证显示两层下半年人均消费最低（11.1 / 18.7 元）、未复购率最高（91.5% / 89.7%），一般保持层下半年总消费仅 1,269 元，继续投入产出极低 |
| 直播渠道强行提客单 | 直播客单价低于店铺是内容场的结构性特征，不宜硬拉；应把直播定位为拉新入口，考核其后续复购转化而非当场客单价 |

### 四、数据局限对建议的约束

- 数据为和鲸社区公开脱敏数据集，字段有限（无成本、无流量、无地域、无退款金额），上述建议的**方向**成立，**具体收益幅度**需在企业真实数据上重新测算
- 无用户行为埋点，无法做漏斗归因，"触达是否有效"只能通过分组对比间接验证
- 仅有一年数据，无法区分季节性波动与趋势性变化，建议至少有两年数据再下长期结论
